# QAOA for MaxCut

The Quantum Approximate Optimization Algorithm is a **hybrid** loop:

- a *p*-layer circuit $U(C,\gamma)U(B,\beta)$ prepares a trial cut
- a classical optimiser updates $(\gamma,\beta)$ to raise the expected cut

We cut the 4-cycle $0-1-2-3-0$. An optimal colouring cuts all 4 edges.

The cost layer is CNOT–RZ–CNOT (not the TSP one-hot QUBO, not the VQC
ansatz).

In [1]:
import math
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

EDGES = ((0, 1), (1, 2), (2, 3), (3, 0))
N = 4
P = 2


def cut_size(bits):
    colors = bits[::-1]
    return sum(colors[i] != colors[j] for i, j in EDGES)

## Cost and mixer unitaries

$U(C,\gamma)=\prod_{(i,j)\in E}\exp(-i\gamma Z_i Z_j)$ and
$U(B,\beta)=\prod_i \exp(-i\beta X_i)$.

In [2]:
def cost(gamma):
    qc = QuantumCircuit(N)
    for i, j in EDGES:
        qc.cx(i, j)
        qc.rz(2 * gamma, j)
        qc.cx(i, j)
    return qc


def mixer(beta):
    qc = QuantumCircuit(N)
    for q in range(N):
        qc.rx(2 * beta, q)
    return qc


def qaoa(params):
    qc = QuantumCircuit(N)
    qc.h(range(N))
    for k in range(P):
        qc.compose(cost(float(params[k])), inplace=True)
        qc.compose(mixer(float(params[P + k])), inplace=True)
    return qc


print(qaoa(np.array([0.3, 0.4, 0.6, 0.2])).draw())

     ┌───┐                                                                     »
q_0: ┤ H ├──■───────────────■──────────────────────────────────────────────────»
     ├───┤┌─┴─┐┌─────────┐┌─┴─┐                     ┌─────────┐                »
q_1: ┤ H ├┤ X ├┤ Rz(0.6) ├┤ X ├──■───────────────■──┤ Rx(1.2) ├────────────────»
     ├───┤└───┘└─────────┘└───┘┌─┴─┐┌─────────┐┌─┴─┐└─────────┘                »
q_2: ┤ H ├─────────────────────┤ X ├┤ Rz(0.6) ├┤ X ├─────■──────────────────■──»
     ├───┤                     └───┘└─────────┘└───┘   ┌─┴─┐   ┌─────────┐┌─┴─┐»
q_3: ┤ H ├─────────────────────────────────────────────┤ X ├───┤ Rz(0.6) ├┤ X ├»
     └───┘                                             └───┘   └─────────┘└───┘»
«     ┌───┐┌─────────┐┌───┐┌─────────┐                                     »
«q_0: ┤ X ├┤ Rz(0.6) ├┤ X ├┤ Rx(1.2) ├──■───────────────■──────────────────»
«     └─┬─┘└─────────┘└─┬─┘└─────────┘┌─┴─┐┌─────────┐┌─┴─┐                »
«q_1: ──┼───────────────┼─────────────┤ 

## Expected cut, maximised by minimising its negation

In [3]:
def expected_cut(params):
    probs = Statevector.from_instruction(qaoa(params)).probabilities_dict()
    return sum(p * cut_size(b) for b, p in probs.items())


def loss(params):
    return -expected_cut(params)


rng = np.random.default_rng(7)
guess = rng.uniform(0, math.pi, size=2 * P)
opt = minimize(loss, guess, method="COBYLA", options={"maxiter": 80, "rhobeg": 0.4})
print("expected cut", expected_cut(opt.x))

probs = Statevector.from_instruction(qaoa(opt.x)).probabilities_dict()
bits, p = max(probs.items(), key=lambda kv: kv[1])
print(f"most likely |{bits}>  colouring={bits[::-1]}  cut={cut_size(bits)}  P={p:.3f}")

expected cut 3.999182027114265
most likely |0101>  colouring=1010  cut=4  P=0.500
